# Estrutura — camada trusted (tancagem)

**Objetivo:** documentar a estrutura do parquet unificado, mapear nulos e analisar tancagem por UF.

**Pré-requisito:** `py estudos/tancagem-abastecimento/pipelines/build_trusted.py`

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import (
    REPO_ROOT,
    TRUSTED_DIR,
    TRUSTED_MANIFEST,
    TRUSTED_PARQUET,
)

EXPECTED_COLS = [
    "Data",
    "NomeEmpresarial",
    "Uf",
    "Municipio",
    "Cnpj",
    "CodInstalacao",
    "Segmento",
    "DetalheInstalacao",
    "Tag",
    "TipoDaUnidade",
    "GrupoDeProdutos",
    "TancagemM3",
]
META_COLS = ["_source_file", "_source_year", "_source_period"]

if not TRUSTED_PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet ausente. Execute build_trusted.py. Esperado: {TRUSTED_PARQUET}"
    )

df = pd.read_parquet(TRUSTED_PARQUET)
manifest = json.loads(TRUSTED_MANIFEST.read_text(encoding="utf-8"))

print(f"Repo: {REPO_ROOT}")
print(f"Parquet: {TRUSTED_PARQUET.relative_to(REPO_ROOT)}")
print(f"Linhas: {len(df):,} | Colunas: {len(df.columns)}")
print(f"Gerado em (manifest): {manifest.get('gerado_em')}")
print(f"Arquivos fonte: {manifest.get('arquivos')}")

Repo: C:\Users\Pichau\Documents\Projetos\ANP\anp-fuel-analytics
Parquet: data\trusted\tancagem-abastecimento\tancagem.parquet
Linhas: 492,412 | Colunas: 15
Gerado em (manifest): 2026-05-23T00:35:30.109163+00:00
Arquivos fonte: 36


## 1. Estrutura do dataset

In [2]:
print("Colunas (ordem no parquet):")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2}. {c} ({df[c].dtype})")

missing_expected = set(EXPECTED_COLS) - set(df.columns)
extra_cols = set(df.columns) - set(EXPECTED_COLS + META_COLS)
print(f"\nColunas esperadas ausentes: {missing_expected or 'nenhuma'}")
print(f"Colunas extras: {extra_cols or 'nenhuma'}")

Colunas (ordem no parquet):
   1. Data (datetime64[us])
   2. NomeEmpresarial (str)
   3. Uf (str)
   4. Municipio (str)
   5. Cnpj (str)
   6. CodInstalacao (str)
   7. Segmento (str)
   8. DetalheInstalacao (str)
   9. Tag (str)
  10. TipoDaUnidade (str)
  11. GrupoDeProdutos (str)
  12. TancagemM3 (float64)
  13. _source_file (str)
  14. _source_year (str)
  15. _source_period (str)

Colunas esperadas ausentes: nenhuma
Colunas extras: nenhuma


In [3]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 492412 entries, 0 to 492411
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Data               492122 non-null  datetime64[us]
 1   NomeEmpresarial    492122 non-null  str           
 2   Uf                 492122 non-null  str           
 3   Municipio          492122 non-null  str           
 4   Cnpj               492122 non-null  str           
 5   CodInstalacao      492122 non-null  str           
 6   Segmento           492122 non-null  str           
 7   DetalheInstalacao  492122 non-null  str           
 8   Tag                492122 non-null  str           
 9   TipoDaUnidade      492122 non-null  str           
 10  GrupoDeProdutos    492122 non-null  str           
 11  TancagemM3         492122 non-null  float64       
 12  _source_file       492412 non-null  str           
 13  _source_year       492412 non-null  str           
 14 

In [4]:
display(df.head(3))
df.describe(include="all").T

,Data,NomeEmpresarial,Uf,Municipio,Cnpj,CodInstalacao,Segmento,DetalheInstalacao,Tag,TipoDaUnidade,GrupoDeProdutos,TancagemM3,_source_file,_source_year,_source_period
0,2022-09-01,2 IRMAOS PRODUTOS DE PETROLEO LTDA,SP,CAMPINAS,43544287000123,1066748,BASES DO RAMO DE TRR,EXCLUSIVA,TQ 01,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,20.0,2022/tancagem_terminais_dados_abertos_2022_09_...,2022,tancagem_terminais_dados_abertos_2022_09_01
1,2022-09-01,2 IRMAOS PRODUTOS DE PETROLEO LTDA,SP,CAMPINAS,43544287000123,1066748,BASES DO RAMO DE TRR,EXCLUSIVA,TQ 02,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,20.0,2022/tancagem_terminais_dados_abertos_2022_09_...,2022,tancagem_terminais_dados_abertos_2022_09_01
2,2022-09-01,2 IRMAOS PRODUTOS DE PETROLEO LTDA,SP,CAMPINAS,43544287000123,1066748,BASES DO RAMO DE TRR,EXCLUSIVA,TQ 05,TANQUE,DERIVADOS E BIOCOMBUSTÍVEIS,20.0,2022/tancagem_terminais_dados_abertos_2022_09_...,2022,tancagem_terminais_dados_abertos_2022_09_01


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Data,492122,NaN,NaN,NaN,2024-04-14 16:52:11.867301,2022-06-01 00:00:00,2023-03-01 00:00:00,2024-01-02 00:00:00,2025-05-09 00:00:00,2026-04-30 00:00:00,NaN
NomeEmpresarial,492122,1233,PETROLEO BRASILEIRO S/A,39064,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Uf,492122,27,SP,136495,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Municipio,492122,814,SANTOS,28323,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cnpj,492122,1915,33000167008862,8882,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CodInstalacao,492122,1919,1032149,8738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segmento,492122,11,BASES DO RAMO DE COMBUSTÍVEIS,90223,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DetalheInstalacao,492122,12,EXCLUSIVA,192741,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tag,492122,6233,01,13441,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TipoDaUnidade,492122,3,TANQUE,443328,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Nulos e preenchimento por coluna

In [5]:
n = len(df)

def empty_string_count(series: pd.Series) -> int:
    if series.dtype != object and not pd.api.types.is_string_dtype(series):
        return 0
    return int(series.fillna("").astype(str).str.strip().eq("").sum())

rows = []
for col in df.columns:
    null_n = int(df[col].isna().sum())
    empty_n = empty_string_count(df[col])
    filled_n = n - null_n
    rows.append(
        {
            "coluna": col,
            "dtype": str(df[col].dtype),
            "nulos": null_n,
            "%_nulos": round(100 * null_n / n, 4),
            "preenchidos": filled_n,
            "%_preenchidos": round(100 * filled_n / n, 4),
            "strings_vazias": empty_n,
            "status": "OK" if null_n == 0 and empty_n == 0 else "revisar",
        }
    )

null_report = pd.DataFrame(rows).set_index("coluna")
null_report

,dtype,nulos,%_nulos,preenchidos,%_preenchidos,strings_vazias,status
coluna,,,,,,,
Data,datetime64[us],290,0.0589,492122,99.9411,0,revisar
NomeEmpresarial,str,290,0.0589,492122,99.9411,290,revisar
Uf,str,290,0.0589,492122,99.9411,290,revisar
Municipio,str,290,0.0589,492122,99.9411,290,revisar
Cnpj,str,290,0.0589,492122,99.9411,290,revisar
CodInstalacao,str,290,0.0589,492122,99.9411,290,revisar
Segmento,str,290,0.0589,492122,99.9411,290,revisar
DetalheInstalacao,str,290,0.0589,492122,99.9411,290,revisar
Tag,str,290,0.0589,492122,99.9411,290,revisar


In [6]:
biz_cols = EXPECTED_COLS
all_biz_null = df[biz_cols].isna().all(axis=1)
print(f"Linhas com todas as colunas de negócio nulas: {all_biz_null.sum():,}")

if all_biz_null.any():
    display(
        df.loc[all_biz_null, META_COLS]
        .value_counts("_source_file")
        .rename("linhas")
        .to_frame()
    )

Linhas com todas as colunas de negócio nulas: 290


,linhas
_source_file,
2025/dezembro.csv,52
2026/abril.csv,52
2026/janeiro.csv,52
2026/marco.csv,52
2022/tancagem_terminais_dados_abertos_2022_09_01.csv,41
2022/tancagem_terminais_dados_abertos_junho_2022.csv,41


In [7]:
partial_null = df[biz_cols].isna().any(axis=1) & ~all_biz_null
print(f"Linhas com nulo parcial (alguma coluna de negócio): {partial_null.sum():,}")

if partial_null.any():
    per_col = df.loc[partial_null, biz_cols].isna().sum().sort_values(ascending=False)
    print("Nulos por coluna (somente linhas parciais):")
    display(per_col[per_col > 0])

Linhas com nulo parcial (alguma coluna de negócio): 0


## 3. Tancagem por UF — último mês e evolução em Goiás

Agregação por UF no **último snapshot** (`Data` máxima):

- **tancagem_m3** — soma de capacidade autorizada
- **cnpj_distintos** — empresas distintas (`Cnpj`)
- **tancagem_por_cnpj** — tancagem média por empresa (m³ / CNPJ)

Para a **evolução temporal**, cada `_source_file` é um snapshot distinto (evita dupla contagem quando vários arquivos compartilham a mesma `Data`).

In [11]:
valid = df.dropna(subset=["Uf", "Cnpj", "TancagemM3"]).copy()

latest_date = valid["Data"].max()
snap_latest = valid[valid["Data"] == latest_date]

print(f"Último snapshot (Data): {latest_date.date()}")
print(f"Arquivos fonte nessa data: {snap_latest['_source_file'].nunique()}")
print(snap_latest["_source_file"].unique())

ranking_uf = (
    snap_latest.groupby("Uf", as_index=False)
    .agg(
        tancagem_m3=("TancagemM3", "sum"),
        cnpj_distintos=("Cnpj", "nunique"),
    )
    .sort_values("tancagem_m3", ascending=False)
    .reset_index(drop=True)
)
ranking_uf.insert(0, "rank", range(1, len(ranking_uf) + 1))
ranking_uf["tancagem_mil_m3"] = (ranking_uf["tancagem_m3"] / 1_000).round(1)
ranking_uf["tancagem_por_cnpj"] = (ranking_uf["tancagem_m3"] / ranking_uf["cnpj_distintos"]).round(1)

display(
    ranking_uf[
        [
            "rank",
            "Uf",
            "tancagem_m3",
            "tancagem_mil_m3",
            "cnpj_distintos",
            "tancagem_por_cnpj",
        ]
    ]
)

Último snapshot (Data): 2026-04-30
Arquivos fonte nessa data: 1
<ArrowStringArray>
['2026/abril.csv']
Length: 1, dtype: str


,rank,Uf,tancagem_m3,tancagem_mil_m3,cnpj_distintos,tancagem_por_cnpj
0,1,SP,23782183.0,23782.2,385,61771.9
1,2,RJ,6105232.0,6105.2,56,109022.0
2,3,PR,4216981.0,4217.0,184,22918.4
3,4,MG,3718091.0,3718.1,130,28600.7
4,5,GO,3391847.0,3391.8,104,32613.9
5,6,RS,3198184.0,3198.2,157,20370.6
6,7,BA,3108931.0,3108.9,73,42588.1
7,8,PE,2639997.0,2640.0,32,82499.9
8,9,MT,1892124.0,1892.1,148,12784.6
9,10,MS,1691862.0,1691.9,80,21148.3


In [13]:
import matplotlib.pyplot as plt

%matplotlib inline

UF_EVOLUCAO = "GO"

go_ev = (
    valid[valid["Uf"] == UF_EVOLUCAO]
    .groupby("_source_file", as_index=False)
    .agg(
        data=("Data", "max"),
        tancagem_m3=("TancagemM3", "sum"),
    )
    .sort_values("data")
    .reset_index(drop=True)
)
go_ev["tancagem_mil_m3"] = (go_ev["tancagem_m3"] / 1_000).round(1)
go_ev["var_pct_snapshot_anterior"] = go_ev["tancagem_m3"].pct_change() * 100

print(f"Evolução da tancagem — {UF_EVOLUCAO} ({len(go_ev)} snapshots)")
display(go_ev)

ax = go_ev.plot(x="data", y="tancagem_m3", kind="line", marker="o", figsize=(10, 4), legend=False)
ax.set_title(f"Tancagem autorizada em {UF_EVOLUCAO} (mil m³)")
ax.set_xlabel("Data do snapshot")
ax.set_ylabel("mil m³")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Evolução da tancagem — GO (36 snapshots)


,_source_file,data,tancagem_m3,tancagem_mil_m3,var_pct_snapshot_anterior
0,2022/tancagem_terminais_dados_abertos_junho_20...,2022-06-01,2.827380e+06,2827.4,NaN
1,2022/tancagem_terminais_dados_abertos_julho_20...,2022-07-01,2.827380e+06,2827.4,0.000000
2,2022/tancagem_terminais_dados_abertos_v1.csv,2022-08-01,2.877380e+06,2877.4,1.768422
3,2022/tancagem_terminais_dados_abertos_2022_09_...,2022-09-01,2.877380e+06,2877.4,0.000000
4,2022/tancagem_terminais_dados_abertos_outubro_...,2022-10-03,2.877369e+06,2877.4,-0.000365
5,2022/tancagem_terminais_dados_abertos_novembro...,2022-11-01,1.889329e+06,1889.3,-34.338325
6,2022/tancagem_terminais_dados_abertos_dezembro...,2022-12-01,1.863082e+06,1863.1,-1.389223
7,2023/janeiro.csv,2023-01-02,2.939092e+06,2939.1,57.754302
8,2023/fevereiro.csv,2023-02-01,2.926548e+06,2926.5,-0.426798
9,2023/marco.csv,2023-03-01,2.932358e+06,2932.4,0.198527


ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.